# Creating Perfume Embeddings

This notebook creates two types of embeddings for perfume data:
1. **Search embeddings** (query→doc): Using `intfloat/multilingual-e5-base` (768 dim)
2. **Recommendation embeddings** (item→item): Using `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` (768 dim)

The embeddings will be saved as parquet files for ingestion into PostgreSQL.

## 1. Setup and Imports

In [19]:
!pip install -U sentence-transformers pyarrow pandas fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.6/687.6 kB 1.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fastparquet]


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cpu


## 2. Load Dataset

In [3]:
# Load the dataset
df = pd.read_csv('../data/processed/dataset_final.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Dataset shape: (24063, 29)

Columns: ['url', 'Perfume', 'Brand_en', 'Country_en', 'Gender_en', 'Rating Value', 'Rating Count', 'Year', 'Top_en', 'Middle_en', 'Base_en', 'Perfumer1', 'Perfumer2', 'mainaccord1_en', 'mainaccord2_en', 'mainaccord3_en', 'mainaccord4_en', 'mainaccord5_en', 'Brand_ru', 'Country_ru', 'Gender_ru', 'Top_ru', 'Middle_ru', 'Base_ru', 'mainaccord1_ru', 'mainaccord2_ru', 'mainaccord3_ru', 'mainaccord4_ru', 'mainaccord5_ru']


,url,Perfume,Brand_en,Country_en,Gender_en,Rating Value,Rating Count,Year,Top_en,Middle_en,...,Country_ru,Gender_ru,Top_ru,Middle_ru,Base_ru,mainaccord1_ru,mainaccord2_ru,mainaccord3_ru,mainaccord4_ru,mainaccord5_ru
0,https://www.fragrantica.com/perfume/xerjoff/ac...,accento-overdose-pride-edition,xerjoff,Italy,unisex,"1,42",201,2022.0,"fruity notes, aldehydes, green notes","bulgarian rose, egyptian jasmine, lily-of-the-...",...,Италия,unisex,"Фруктовые ноты, альдегиды, зелёные ноты","Булгарская роза, египтианская жасмина, лилия-в...","Эвкалипт, сосна",роза,вуаля,Фрукты,ароматические,Цветок
1,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2024,jean-paul-gaultier,France,women,"1,86",70,2024.0,"yuzu, citruses","orange blossom, neroli",...,Франция,women,"Юзу, цитрусовые","апельсиновый цветок, нероли","мускус, блондинка",цитрусовые,белый цветок,сладкий,свежие,Муски
2,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2023,jean-paul-gaultier,France,unisex,"1,91",285,2023.0,"blood orange, yuzu","neroli, orange blossom",...,Франция,unisex,"Кровь оранжевая, юдзу","Нероли, оранжевый цветок","мускус, белый лес",цитрусовые,белый цветок,сладкий,свежее острое,Муски
3,https://www.fragrantica.com/perfume/bruno-bana...,pride-edition-man,bruno-banani,Germany,men,"1,92",59,2019.0,"guarana, grapefruit, red apple","walnut, lavender, guava",...,Германия,men,"Гуарана, грейпфруты, красное яблоко","Орех, лаванда, гуава","Ветер, бензуин, автожелтый",Фрукты,Сумасшедший,вуаля,тропические,NaN
4,https://www.fragrantica.com/perfume/jean-paul-...,le-male-pride-collector,jean-paul-gaultier,France,men,"1,93",632,2020.0,"mint, lavender, cardamom, artemisia, bergamot","caraway, cinnamon, orange blossom",...,Франция,men,"Мята, лаванда, кардамом, артемизия, бергамот",":: автомобиль, корица, цветок оранжевого цвета","ваниль, сандальная древесина, автожелтый, кедр...",ароматические,тёплый,свежее острое,Корица,ванильный


## 3. Prepare Text for Embeddings

We'll create rich text descriptions combining:
- Perfume name
- Brand (English)
- Gender
- Top, Middle, Base notes (English)
- Main accords (English)

In [4]:
def create_search_text(row):
    """Create a rich text description for search embeddings."""
    parts = []
    
    # Perfume name
    if pd.notna(row['Perfume']):
        parts.append(f"Perfume: {row['Perfume']}")
    
    # Brand
    if pd.notna(row['Brand_en']):
        parts.append(f"Brand: {row['Brand_en']}")
    
    # Gender
    if pd.notna(row['Gender_en']):
        parts.append(f"Gender: {row['Gender_en']}")
    
    # Notes
    notes = []
    if pd.notna(row['Top_en']):
        notes.append(f"Top notes: {row['Top_en']}")
    if pd.notna(row['Middle_en']):
        notes.append(f"Middle notes: {row['Middle_en']}")
    if pd.notna(row['Base_en']):
        notes.append(f"Base notes: {row['Base_en']}")
    if notes:
        parts.extend(notes)
    
    # Main accords
    accords = []
    for i in range(1, 6):
        accord_col = f'mainaccord{i}_en'
        if accord_col in row and pd.notna(row[accord_col]):
            accords.append(row[accord_col])
    if accords:
        parts.append(f"Accords: {', '.join(accords)}")
    
    return ". ".join(parts)


def create_rec_text(row):
    """Create text description for recommendation embeddings - only notes and accords."""
    parts = []
    
    # Notes
    notes = []
    if pd.notna(row['Top_en']):
        notes.append(f"Top notes: {row['Top_en']}")
    if pd.notna(row['Middle_en']):
        notes.append(f"Middle notes: {row['Middle_en']}")
    if pd.notna(row['Base_en']):
        notes.append(f"Base notes: {row['Base_en']}")
    if notes:
        parts.extend(notes)
    
    # Main accords
    accords = []
    for i in range(1, 6):
        accord_col = f'mainaccord{i}_en'
        if accord_col in row and pd.notna(row[accord_col]):
            accords.append(row[accord_col])
    if accords:
        parts.append(f"Accords: {', '.join(accords)}")
    
    return ". ".join(parts)


# Create text descriptions
df['search_text'] = df.apply(create_search_text, axis=1)
df['rec_text'] = df.apply(create_rec_text, axis=1)

print("Sample SEARCH text descriptions:")
for i in range(min(2, len(df))):
    print(f"\n{i+1}. {df['search_text'].iloc[i][:200]}...")

print("\n" + "="*80)
print("Sample RECOMMENDATION text descriptions (notes + accords only):")
for i in range(min(2, len(df))):
    print(f"\n{i+1}. {df['rec_text'].iloc[i][:200]}...")

Sample SEARCH text descriptions:

1. Perfume: accento-overdose-pride-edition. Brand: xerjoff. Gender: unisex. Top notes: fruity notes, aldehydes, green notes. Middle notes: bulgarian rose, egyptian jasmine, lily-of-the-valley. Base notes...

2. Perfume: classique-pride-2024. Brand: jean-paul-gaultier. Gender: women. Top notes: yuzu, citruses. Middle notes: orange blossom, neroli. Base notes: musk, blonde woods. Accords: citrus, white floral,...

Sample RECOMMENDATION text descriptions (notes + accords only):

1. Top notes: fruity notes, aldehydes, green notes. Middle notes: bulgarian rose, egyptian jasmine, lily-of-the-valley. Base notes: eucalyptus, pine. Accords: rose, woody, fruity, aromatic, floral...

2. Top notes: yuzu, citruses. Middle notes: orange blossom, neroli. Base notes: musk, blonde woods. Accords: citrus, white floral, sweet, fresh, musky...


## 4. Generate Search Embeddings (multilingual-e5-base)

For search/retrieval purposes - user queries to perfume documents.

In [5]:
# Load E5 model for search
print("Loading multilingual-e5-base model...")
search_model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)

print(f"Model loaded on {device}")
print(f"Embedding dimension: {search_model.get_sentence_embedding_dimension()}")

Loading multilingual-e5-base model...


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded on cpu
Embedding dimension: 768


In [6]:
# Generate search embeddings
# E5 models require "passage: " prefix for documents
texts_with_prefix = ["passage: " + text for text in df['search_text'].tolist()]

print("Generating search embeddings...")
search_embeddings = search_model.encode(
    texts_with_prefix,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # Normalize for cosine similarity
)

print(f"Search embeddings shape: {search_embeddings.shape}")

Generating search embeddings...


Batches:   0%|          | 0/752 [00:00<?, ?it/s]

Search embeddings shape: (24063, 768)


## 5. Generate Recommendation Embeddings (paraphrase-multilingual-mpnet-base-v2)

For item-to-item similarity and recommendations.

In [7]:
# Load paraphrase model for recommendations
print("Loading paraphrase-multilingual-mpnet-base-v2 model...")
rec_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2', device=device)

print(f"Model loaded on {device}")
print(f"Embedding dimension: {rec_model.get_sentence_embedding_dimension()}")

Loading paraphrase-multilingual-mpnet-base-v2 model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded on cpu
Embedding dimension: 768


In [8]:
# Generate recommendation embeddings
# Paraphrase models don't need special prefixes
# Using only notes and accords for better similarity matching
print("Generating recommendation embeddings...")
rec_embeddings = rec_model.encode(
    df['rec_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # Normalize for cosine similarity
)

print(f"Recommendation embeddings shape: {rec_embeddings.shape}")

Generating recommendation embeddings...


Batches:   0%|          | 0/752 [00:00<?, ?it/s]

Recommendation embeddings shape: (24063, 768)


## 6. Prepare Data for Export

Combine original data with embeddings.

In [23]:
# Create a copy of the dataframe with embeddings
df_export = df.copy()

# Convert embeddings to proper format for parquet
# Store as numpy arrays to avoid pyarrow issues
df_export['search_embedding'] = list(search_embeddings)
df_export['rec_embedding'] = list(rec_embeddings)

print(f"Export dataframe shape: {df_export.shape}")
print(f"Columns: {list(df_export.columns)}")

Export dataframe shape: (24063, 33)
Columns: ['url', 'Perfume', 'Brand_en', 'Country_en', 'Gender_en', 'Rating Value', 'Rating Count', 'Year', 'Top_en', 'Middle_en', 'Base_en', 'Perfumer1', 'Perfumer2', 'mainaccord1_en', 'mainaccord2_en', 'mainaccord3_en', 'mainaccord4_en', 'mainaccord5_en', 'Brand_ru', 'Country_ru', 'Gender_ru', 'Top_ru', 'Middle_ru', 'Base_ru', 'mainaccord1_ru', 'mainaccord2_ru', 'mainaccord3_ru', 'mainaccord4_ru', 'mainaccord5_ru', 'search_text', 'rec_text', 'search_embedding', 'rec_embedding']


## 7. Save to Parquet

Save the data with embeddings in parquet format for efficient storage and PostgreSQL ingestion.

In [24]:
# Save to parquet
output_path = '../data/processed/perfumes_with_embeddings.parquet'


# Convert embeddings to list of lists for parquet
df_export_copy = df_export.copy()
df_export_copy['search_embedding'] = df_export_copy['search_embedding'].apply(lambda x: x.tolist() if hasattr(x, 'tolist') else x)
df_export_copy['rec_embedding'] = df_export_copy['rec_embedding'].apply(lambda x: x.tolist() if hasattr(x, 'tolist') else x)

# Save with fastparquet to avoid pyarrow conflicts in the same kernel session
df_export_copy.to_parquet(output_path, index=False, engine='fastparquet')

print(f"✅ Data saved to: {output_path}")

# Get file size
import os
file_size = os.path.getsize(output_path)
print(f"File size: {file_size / 1024 / 1024:.2f} MB")

print(f"\n✅ Export complete!")
print(f"Saved {len(df_export_copy):,} perfumes with embeddings")
print(f"- Search embeddings: 768-dim (multilingual-e5-base)")
print(f"- Recommendation embeddings: 768-dim (paraphrase-multilingual-mpnet-base-v2)")
print(f"\nTo verify the file, restart the kernel and run:")
print(f"  df = pd.read_parquet('{output_path}')")
print(f"  print(f'Rows: {{len(df)}}, Columns: {{len(df.columns)}}')")
print(f"  print(f'Search emb dim: {{len(df[\"search_embedding\"].iloc[0])}}')")
print(f"  print(f'Rec emb dim: {{len(df[\"rec_embedding\"].iloc[0])}}')")

✅ Data saved to: ../data/processed/perfumes_with_embeddings.parquet
File size: 557.59 MB

✅ Export complete!
Saved 24,063 perfumes with embeddings
- Search embeddings: 768-dim (multilingual-e5-base)
- Recommendation embeddings: 768-dim (paraphrase-multilingual-mpnet-base-v2)

To verify the file, restart the kernel and run:
  df = pd.read_parquet('../data/processed/perfumes_with_embeddings.parquet')
  print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
  print(f'Search emb dim: {len(df["search_embedding"].iloc[0])}')
  print(f'Rec emb dim: {len(df["rec_embedding"].iloc[0])}')


## 8. Summary

**Embeddings created:**
- **Search embeddings**: 768-dimensional vectors using `intfloat/multilingual-e5-base`
  - Purpose: Query-to-document search
  - Use case: User searches for perfumes by description
  
- **Recommendation embeddings**: 768-dimensional vectors using `paraphrase-multilingual-mpnet-base-v2`
  - Purpose: Item-to-item similarity
  - Use case: "Similar perfumes" recommendations

**Output:**
- File: `data/processed/perfumes_with_embeddings.parquet`
- Ready for PostgreSQL ingestion with pgvector